In [5]:
import asyncio
import time

# Task 1 — Pure Synchronous Execution

In [8]:
def run_sync_tasks(name):
    print(f"{name} Started")
    print(f"{name} Delay")
    time.sleep(2)
    print(f"{name} Finished") 

In [21]:
start = time.time()
run_sync_tasks("task1")
run_sync_tasks("task2")
run_sync_tasks("task3")
run_sync_tasks("task4")
run_sync_tasks("task5")
run_sync_tasks("task6")
run_sync_tasks("task7")
run_sync_tasks("task8")

print(f"Total time = {time.time() - start}")

task1 Started
task1 Delay
task1 Finished
task2 Started
task2 Delay
task2 Finished
task3 Started
task3 Delay
task3 Finished
task4 Started
task4 Delay
task4 Finished
task5 Started
task5 Delay
task5 Finished
task6 Started
task6 Delay
task6 Finished
task7 Started
task7 Delay
task7 Finished
task8 Started
task8 Delay
task8 Finished
Total time = 16.01045823097229


# Task 2 — Async Sequential (Without gather)

In [12]:
async def run_async_tasks(name):
    print(f"{name} Started")
    await asyncio.sleep(2)
    print(f"{name} Finished")     

In [20]:
async def main():
    start = time.time()
    for i in range(1,9):
        await run_async_tasks(f"task {i}")
    print(f"Total time = {time.time() - start}")

await main()

task 1 Started
task 1 Finished
task 2 Started
task 2 Finished
task 3 Started
task 3 Finished
task 4 Started
task 4 Finished
task 5 Started
task 5 Finished
task 6 Started
task 6 Finished
task 7 Started
task 7 Finished
task 8 Started
task 8 Finished
Total time = 16.06085705757141



# Task 3 — Async Concurrent Using gather()


In [23]:
async def run_async_tasks_gather(name):
    print(f"{name} Started")
    await asyncio.sleep(2)
    print(f"{name} Finished")    

In [26]:
async def main():
    tasks = []
    for i in range(1,9):
        tasks.append(run_async_tasks_gather(f"tasks{i}"))

    start = time.time()
    
    await asyncio.gather(*tasks)
    
    print(f"Total time = {time.time() - start}")

await main()


tasks1 Started
tasks2 Started
tasks3 Started
tasks4 Started
tasks5 Started
tasks6 Started
tasks7 Started
tasks8 Started
tasks1 Finished
tasks3 Finished
tasks7 Finished
tasks6 Finished
tasks8 Finished
tasks5 Finished
tasks2 Finished
tasks4 Finished
Total time = 2.0008509159088135


# Task 4 — Blocking CPU Inside Event Loop

In [6]:
def heavy_cpu_function():
    print("Heavy CPU Started")
    result = 0
    for i in range(100_000_000):
        result += i * i
    print("Heavy CPU Finished")
    return result

In [30]:
async def async_task():
    for i in range(5):
        print(f"Async Task Running: {i + 1}")
        await asyncio.sleep(1)    

In [32]:
async def main():
    start = time.time()
    task = asyncio.create_task(async_task())
    heavy_cpu_function()
    await task
    print(f"Total time = {time.time() - start}")

await main()

Heavy CPU Started
Heavy CPU Finished
Async Task Running: 1
Async Task Running: 2
Async Task Running: 3
Async Task Running: 4
Async Task Running: 5
Total time = 9.816894292831421


# Task 5 — Yielding Inside Single Thread

In [1]:
async def task1():
    for i in range(10):
        print(f"Task 1: {i}")
        await asyncio.sleep(0)

In [2]:
async def task2():
    for i in range(10):
        print(f"Task 2: {i}")
        await asyncio.sleep(0)

In [7]:
async def main():
    await asyncio.gather(task1(), task2())

await main()

Task 1: 0
Task 2: 0
Task 1: 1
Task 2: 1
Task 1: 2
Task 2: 2
Task 1: 3
Task 2: 3
Task 1: 4
Task 2: 4
Task 1: 5
Task 2: 5
Task 1: 6
Task 2: 6
Task 1: 7
Task 2: 7
Task 1: 8
Task 2: 8
Task 1: 9
Task 2: 9


# Task 6 — Using asyncio.to_thread()

In [9]:
def heavy_cpu():
    print("Heavy CPU task started")

    result = 0
    for i in range(50_000_000):
        result += i

    print("Heavy CPU task finished")
    return result

In [10]:
async def light_task():
    for i in range(5):
        print(f"Light task: {i}")
        await asyncio.sleep(1)

In [11]:
async def main():
    heavy_task = asyncio.to_thread(heavy_cpu)

    await asyncio.gather(
        heavy_task,
        light_task()
    )


await main()

Heavy CPU task started
Light task: 0
Light task: 1
Heavy CPU task finished
Light task: 2
Light task: 3
Light task: 4


# Task 7 — Race Condition Simulation

In [16]:
import threading
shared_counter = 0

In [14]:
def increment_without_lock():
    global shared_counter

    for _ in range(100000):
        temp = shared_counter
        time.sleep(0)
        shared_counter = temp + 1

In [15]:
def run_without_lock():
    global shared_counter
    shared_counter = 0

    threads = []

    for _ in range(5):
        t = threading.Thread(target=increment_without_lock)
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print("Without Lock:", shared_counter)


In [17]:
lock = threading.Lock()

def increment_with_lock():
    global shared_counter

    for _ in range(100000):
        with lock:
            shared_counter += 1


In [18]:
def run_with_lock():
    global shared_counter
    shared_counter = 0

    threads = []

    for _ in range(5):
        t = threading.Thread(target=increment_with_lock)
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print("With Lock:", shared_counter)

In [19]:
if __name__ == "__main__":
    run_without_lock()
    run_with_lock()

Without Lock: 100299
With Lock: 500000


# Task 8 — Async + Thread Synchronization Hybrid

In [20]:
shared_counter = 0
lock = threading.Lock()

In [21]:
def cpu_work():
    global shared_counter

    for _ in range(100000):
        with lock:
            shared_counter += 1

    return shared_counter

In [22]:
async def async_task(task_id):
    print(f"Task {task_id} started")

    result = await asyncio.to_thread(cpu_work)

    print(f"Task {task_id} finished")

    return result

In [23]:
async def main():
    global shared_counter

    tasks = [
        async_task(1),
        async_task(2),
        async_task(3),
        async_task(4),
        async_task(5)
    ]

    results = await asyncio.gather(*tasks)

    print("\nAll tasks completed")
    print("Shared Counter:", shared_counter)


await main()

Task 1 started
Task 2 started
Task 3 started
Task 4 started
Task 5 started
Task 2 finished
Task 1 finished
Task 3 finished
Task 5 finished
Task 4 finished

All tasks completed
Shared Counter: 500000
